# FaceForge — Phase 6 Notebook

**Status:** Phases 0–5 complete. This notebook starts from the Phase 5 best checkpoint
and runs the Phase 6 corrected fine-tune.

**Key fix over Phase 6B:** Phase 6B dropped `perc_weight` from 1.0 → 0.1, which removed
90 % of the sharpness gradient and let MSE-optimal blurry outputs dominate. Phase 6 keeps
`perc_weight = 1.0` and only raises `lambda_aux` 2.0 → 15.0, shifting attribute gradient
from ~4 % → ~23 % of total while preserving sharpness.

```
total = 0.5·MSE  +  1.0·perc  +  0.05·KL_free_bits  +  15.0·aux
```

**Run order:** Execute every section in order. Section 4 (training) is resumable — just
re-run the training cell and it picks up where it left off.

## Section 0 — Setup & Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q ftfy clip-anytorch scikit-image matplotlib

In [ ]:
import os, subprocess, shutil, zipfile, sys
import torch, torchvision
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# ── Paths ────────────────────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/FaceForge'
CKPT_ROOT    = os.path.join(DRIVE_ROOT, 'checkpoints')
REPO_ROOT    = '/content/FaceForge'
DRIVE_DATA   = '/content/drive/MyDrive/FaceForge/data/celeba'
LOCAL_DATA   = '/content/celeba_local'
LOCAL_CELEBA = os.path.join(LOCAL_DATA, 'celeba')

# ── Data: copy from Drive to local SSD once per session ──────────────────
if not os.path.exists(os.path.join(LOCAL_CELEBA, 'img_align_celeba')):
    os.makedirs(LOCAL_CELEBA, exist_ok=True)
    print('Copying zip from Drive to local SSD...')
    shutil.copy(os.path.join(DRIVE_DATA, 'img_align_celeba.zip'),
                '/content/img_align_celeba.zip')
    print('Extracting...')
    with zipfile.ZipFile('/content/img_align_celeba.zip', 'r') as z:
        z.extractall(LOCAL_CELEBA)
    os.remove('/content/img_align_celeba.zip')
    for f in ['list_attr_celeba.txt', 'identity_CelebA.txt',
              'list_bbox_celeba.txt', 'list_landmarks_align_celeba.txt',
              'list_eval_partition.txt']:
        src_f = os.path.join(DRIVE_DATA, f)
        if os.path.exists(src_f):
            shutil.copy(src_f, os.path.join(LOCAL_CELEBA, f))
    print(f'Done. Data ready at {LOCAL_DATA}')
else:
    print('Local data already ready.')

DATA_ROOT = LOCAL_DATA
os.makedirs(CKPT_ROOT, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Data root  : {DATA_ROOT}')
print(f'Ckpt root  : {CKPT_ROOT}')
print(f'Device     : {device}')
print(f'PyTorch    : {torch.__version__}')
print(f'CUDA       : {torch.version.cuda}')
print(f'GPU        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Sync the latest code from GitHub (picks up any fixes pushed since last session)
REPO_URL = 'https://github.com/janampatel/FaceForge.git'
if os.path.isdir(REPO_ROOT):
    print('Repo exists — pulling latest')
    os.system(f'git -C {REPO_ROOT} pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_ROOT}')
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('Repo ready.')

## Section 1 — DataLoader (128×128)

In [ ]:
from data_pipeline import make_dataloader, SELECTED_ATTRS, NUM_ATTRS

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

train_loader = make_dataloader(
    root=DATA_ROOT, split='train', download=False,
    image_size=128, batch_size=64, num_workers=4,
    pin_memory=True, prefetch_factor=4, persistent_workers=True,
)
val_loader = make_dataloader(
    root=DATA_ROOT, split='valid', download=False,
    image_size=128, batch_size=8, num_workers=2, pin_memory=True,
)

imgs, attrs = next(iter(train_loader))
print(f'Train batch : {imgs.shape}  attrs: {attrs.shape}  '
      f'range [{imgs.min():.2f}, {imgs.max():.2f}]')
assert imgs.shape[2] == 128 and attrs.shape[1] == 18

## Section 2 — Models & Losses

In [ ]:
from full_cvae_v3 import (
    FullCVAEv3, AuxAttrLoss, FREE_BITS,
    save_checkpoint, load_checkpoint,
    attribute_swap_grid, SWAP_ATTRS,
    LATENT_DIM, PHOTO_EMB_DIM, IMG_SIZE,
)
from clip_probe import load_clip, load_probe
from losses import PerceptualLoss
from data_pipeline import denormalize

# Run VGG perceptual loss at 128x128 instead of 224x224 — 3-4x faster, avoids OOM
def _preprocess_128(self, x):
    x = (x + 1.0) / 2.0
    return (x - self.mean) / self.std
PerceptualLoss._preprocess = _preprocess_128

PROBE_PATH = os.path.join(CKPT_ROOT, 'clip_probe.pt')
clip_model, _ = load_clip(device)
probe         = load_probe(PROBE_PATH, device)
for p in list(clip_model.parameters()) + list(probe.parameters()):
    p.requires_grad = False
print('CLIP + probe loaded and frozen.')

## Section 3 — Phase 5 Checkpoint Sanity Check

Load the Phase 5 best model, confirm it produces sharp 128×128 images.
The Phase 6A amplification swap (val=3.0) is shown here as a reference baseline
— Phase 6 should match this quality with val=1.0.

In [ ]:
P5_BEST  = os.path.join(CKPT_ROOT, 'full_cvae_v3_best.pt')

p5_model = FullCVAEv3(beta=0.1).to(device)
ckpt5    = torch.load(P5_BEST, map_location=device, weights_only=False)
p5_model.load_state_dict(ckpt5['model'])
p5_model.eval()
print(f'Phase 5 best loaded  (epoch {ckpt5["epoch"]})')

In [ ]:
# Reconstruction check
N = 8
v_imgs, v_attrs = next(iter(val_loader))
v_imgs = v_imgs.to(device); v_attrs = v_attrs.to(device)
with torch.no_grad():
    p5_recon, _, _ = p5_model(v_imgs, v_attrs)

fig, axes = plt.subplots(2, N, figsize=(N * 2.5, 6))
fig.suptitle('Phase 5 best — Top: real   Bottom: reconstruction', fontsize=10)
for i in range(N):
    axes[0, i].imshow(denormalize(v_imgs[i].cpu())); axes[0, i].axis('off')
    axes[1, i].imshow(denormalize(p5_recon[i].cpu())); axes[1, i].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT, 'phase5_reconstructions.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Phase 6A reference: attribute swap at val=3.0 (amplification baseline)
N_DEMO = 4
d_imgs, d_attrs = next(iter(val_loader))
swap_ref = [
    ('+ Smile [3.0]',      SWAP_ATTRS['Smiling'],    3.0),
    ('+ Glasses [3.0]',    SWAP_ATTRS['Eyeglasses'], 3.0),
    ('+ Bangs [3.0]',      SWAP_ATTRS['Bangs'],      3.0),
]
grid_ref   = attribute_swap_grid(p5_model, d_imgs[:N_DEMO], d_attrs[:N_DEMO], swap_ref, device)
col_labels = ['Recon'] + [s[0] for s in swap_ref]

fig, axes = plt.subplots(N_DEMO, len(col_labels), figsize=(len(col_labels)*2.8, N_DEMO*3.2))
fig.suptitle('Phase 5 reference swap (val=3.0 amplification) — Phase 6 target: same quality at val=1.0', fontsize=9)
for r in range(N_DEMO):
    for c in range(len(col_labels)):
        axes[r, c].imshow(denormalize(grid_ref[r, c].cpu()))
        axes[r, c].axis('off')
        if r == 0: axes[r, c].set_title(col_labels[c], fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT, 'phase5_ref_swap.png'), dpi=120, bbox_inches='tight')
plt.show()

## Section 4 — Phase 6: Corrected Fine-tune

Loss formula:
```
total = 0.5·MSE  +  1.0·perc  +  0.05·KL_free_bits  +  15.0·aux
```

Key changes vs Phase 5:
- `lambda_aux`: 2.0 → **15.0** — attributes get 23% of gradient (was 4%)
- `perc_weight`: stays **1.0** — sharpness fully preserved (Phase 6B mistake was dropping this to 0.1)
- `beta`: 0.1 → **0.05** — decoder slightly more freedom to change appearance
- `lr`: 1e-4 → **1e-5** — prevents catastrophic forgetting of sharpness
- Starts from `full_cvae_v3_best.pt`, saves to `full_cvae_v3_p6_best.pt`

In [ ]:
# Hyperparameters
P6_EPOCHS      = 20
P6_LR          = 1e-5       # very conservative — prevents catastrophic forgetting
P6_PERC_WEIGHT = 1.0        # KEY: same as Phase 5, NOT 0.1 (Phase 6B mistake)
P6_LAM_AUX     = 15.0       # was 2.0 in Phase 5 → aux is now ~23% of gradient budget
P6_BETA        = 0.05       # fixed — no annealing needed, model already converged
P6_FREE_BITS   = 0.5        # unchanged from Phase 5
P6_BATCH       = 64
P6_WORKERS     = 4

P6_BASE_CKPT  = os.path.join(CKPT_ROOT, 'full_cvae_v3_best.pt')    # Phase 5 best
P6_CKPT       = os.path.join(CKPT_ROOT, 'full_cvae_v3_p6.pt')      # Phase 6 latest
P6_BEST_CKPT  = os.path.join(CKPT_ROOT, 'full_cvae_v3_p6_best.pt') # Phase 6 best

In [ ]:
from full_cvae_v3 import train_one_epoch

# Corrected loss instances
perc_loss_p6 = PerceptualLoss(weight=P6_PERC_WEIGHT).to(device)
perc_loss_p6.eval()
aux_loss_p6  = AuxAttrLoss(clip_model, probe, lam=P6_LAM_AUX).to(device)

p6_model = FullCVAEv3(beta=P6_BETA).to(device)
p6_opt   = torch.optim.Adam(p6_model.parameters(), lr=P6_LR, betas=(0.5, 0.999))
p6_scaler = torch.amp.GradScaler('cuda')

p6_start    = 0
p6_hist     = {'total': [], 'recon': [], 'kl': [], 'aux': [], 'perc': []}
p6_best_val = float('inf')

if os.path.exists(P6_CKPT):
    p6_start, p6_hist = load_checkpoint(P6_CKPT, p6_model, p6_opt, device)
    for g in p6_opt.param_groups:
        g['initial_lr'] = P6_LR
    p6_best_val = min(p6_hist['total']) if p6_hist['total'] else float('inf')
    print(f'Resuming Phase 6 from epoch {p6_start}')
else:
    ckpt = torch.load(P6_BASE_CKPT, map_location=device, weights_only=False)
    p6_model.load_state_dict(ckpt['model'])
    print('Starting Phase 6 from Phase 5 best checkpoint')

p6_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    p6_opt, T_max=P6_EPOCHS, eta_min=1e-7,
    last_epoch=p6_start - 1 if p6_start > 0 else -1,
)
print(f'Params: {sum(p.numel() for p in p6_model.parameters() if p.requires_grad):,}')

In [ ]:
p6_train_loader = make_dataloader(
    root=DATA_ROOT, split='train', download=False,
    image_size=128, batch_size=P6_BATCH, num_workers=P6_WORKERS,
    pin_memory=True, prefetch_factor=4, persistent_workers=True,
)

for epoch in range(p6_start, P6_EPOCHS):
    current_lr = p6_opt.param_groups[0]['lr']

    m = train_one_epoch(
        p6_model, p6_train_loader, p6_opt, device,
        aux_loss_fn  = aux_loss_p6,
        perc_loss_fn = perc_loss_p6,
        beta         = P6_BETA,
        free_bits    = P6_FREE_BITS,
        scaler       = p6_scaler,
    )
    p6_sched.step()

    for k in ['total', 'recon', 'kl', 'aux', 'perc']:
        p6_hist[k].append(m[k])

    is_best = m['total'] < p6_best_val
    if is_best:
        p6_best_val = m['total']
        save_checkpoint(p6_model, p6_opt, epoch + 1, p6_hist, P6_BEST_CKPT,
                        extra={'perc_weight': P6_PERC_WEIGHT,
                               'lam_aux': P6_LAM_AUX, 'beta': P6_BETA})
    save_checkpoint(p6_model, p6_opt, epoch + 1, p6_hist, P6_CKPT,
                    extra={'perc_weight': P6_PERC_WEIGHT,
                           'lam_aux': P6_LAM_AUX, 'beta': P6_BETA})

    print(f'Epoch [{epoch+1:02d}/{P6_EPOCHS}]  '
          f'total={m["total"]:.4f}  mse={m["recon"]:.4f}  '
          f'perc={m["perc"]:.4f}  kl={m["kl"]:.4f}  '
          f'aux={m["aux"]:.4f}  lr={current_lr:.2e}'
          + (' <- best' if is_best else ''))

print('Phase 6 training complete.')

## Section 5 — Evaluation

### Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
keys   = ['total', 'recon', 'perc', 'kl', 'aux']
titles = ['Total', 'Recon (MSE)', 'Perceptual (VGG)', 'KL', 'Aux (Attr BCE)']
for ax, key, title in zip(axes, keys, titles):
    vals = p6_hist.get(key, [])
    ax.plot(range(1, len(vals) + 1), vals, marker='o', markersize=3)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)
fig.suptitle('Phase 6 Loss Curves  (perc_weight=1.0, lambda_aux=15.0)', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT, 'phase6_loss_curves.png'), dpi=120, bbox_inches='tight')
plt.show()

### Reconstruction Quality

Phase 5 best vs Phase 6 best on the same validation images. Sharpness should be identical; attributes in the swap below should be clearer.

In [ ]:
p6_best_model = FullCVAEv3(beta=P6_BETA).to(device)
ckpt_p6b      = torch.load(P6_BEST_CKPT, map_location=device, weights_only=False)
p6_best_model.load_state_dict(ckpt_p6b['model'])
p6_best_model.eval()
print(f'Phase 6 best loaded  (epoch {ckpt_p6b["epoch"]})')

In [ ]:
N = 6
v_imgs, v_attrs = next(iter(val_loader))
v_imgs = v_imgs[:N].to(device); v_attrs = v_attrs[:N].to(device)

with torch.no_grad():
    p5_r, _, _ = p5_model(v_imgs, v_attrs)
    p6_r, _, _ = p6_best_model(v_imgs, v_attrs)

fig, axes = plt.subplots(3, N, figsize=(N * 2.5, 9))
fig.suptitle('Row 1: Real   Row 2: Phase 5 recon   Row 3: Phase 6 recon', fontsize=10)
for r, (row_imgs, label) in enumerate(zip([v_imgs, p5_r, p6_r],
                                          ['Real', 'Phase 5', 'Phase 6'])):
    for c in range(N):
        axes[r, c].imshow(denormalize(row_imgs[c].cpu()))
        axes[r, c].axis('off')
        if c == 0: axes[r, c].set_ylabel(label, fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT, 'phase6_reconstructions.png'), dpi=120, bbox_inches='tight')
plt.show()

### Attribute Swap at val=1.0 (Primary Success Criterion)

**Success:** Smile, Eyeglasses, Bangs visible at val=1.0 without amplification.
Phase 6A needed val=3.0 to see these changes — Phase 6 fine-tuning should eliminate that need.

In [ ]:
N_DEMO = 4
d_imgs, d_attrs = next(iter(val_loader))

swap_p6 = [
    ('+ Smile',   SWAP_ATTRS['Smiling'],    1.0),
    ('- Smile',   SWAP_ATTRS['Smiling'],    0.0),
    ('+ Glasses', SWAP_ATTRS['Eyeglasses'], 1.0),
    ('+ Bangs',   SWAP_ATTRS['Bangs'],      1.0),
    ('+ Blond',   SWAP_ATTRS['Blond_Hair'], 1.0),
]
col_labels = ['Recon'] + [s[0] for s in swap_p6]

grid_p6 = attribute_swap_grid(
    p6_best_model, d_imgs[:N_DEMO], d_attrs[:N_DEMO], swap_p6, device)

fig, axes = plt.subplots(N_DEMO, len(col_labels),
                         figsize=(len(col_labels)*2.8, N_DEMO*3.2))
fig.suptitle('Phase 6 attr swap  val=1.0  (no amplification needed)', fontsize=10)
for r in range(N_DEMO):
    for c in range(len(col_labels)):
        axes[r, c].imshow(denormalize(grid_p6[r, c].cpu()))
        axes[r, c].axis('off')
        if r == 0: axes[r, c].set_title(col_labels[c], fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_ROOT, 'phase6_attr_swap.png'), dpi=120, bbox_inches='tight')
plt.show()

### Validation Checks

In [ ]:
print('Phase 6 Validation:')

assert os.path.exists(P6_CKPT) and os.path.exists(P6_BEST_CKPT), 'Missing checkpoints'
print('[OK] Checkpoints saved')

n_trained = len(p6_hist['total'])
assert n_trained > 0
print(f'[OK] Trained {n_trained} epochs')

if p6_hist['perc']:
    drift = p6_hist['perc'][-1] / p6_hist['perc'][0]
    tag = '[OK]' if drift < 1.1 else '[WARN] Perceptual rising — check sharpness'
    print(f'{tag} Perceptual: {p6_hist["perc"][0]:.4f} -> {p6_hist["perc"][-1]:.4f}')

if p6_hist['aux']:
    tag = '[OK]' if p6_hist['aux'][-1] < p6_hist['aux'][0] else '[WARN] Aux not decreasing'
    print(f'{tag} Aux:         {p6_hist["aux"][0]:.4f} -> {p6_hist["aux"][-1]:.4f}')

p6_best_model.eval()
dummy_x = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE, device=device)
dummy_a = torch.zeros(2, 18, device=device)
with torch.no_grad():
    r_out, mu_out, _ = p6_best_model(dummy_x, dummy_a)
assert r_out.shape == (2, 3, IMG_SIZE, IMG_SIZE)
assert r_out.min() >= -1.01 and r_out.max() <= 1.01
print(f'[OK] Forward pass: {list(dummy_x.shape)} -> {list(r_out.shape)}')

for f in ['phase6_loss_curves.png', 'phase6_reconstructions.png', 'phase6_attr_swap.png']:
    assert os.path.exists(os.path.join(DRIVE_ROOT, f)), f'Missing: {f}'
print('[OK] All visualisation files saved to Drive')
print('\nPhase 6 complete. Proceed to Phase 7 (Evaluation Suite).')